# 4 · Approval and risk strategy

Illustrative approval policies use **validation prediction thresholds only**, then report the same fixed thresholds on the untouched test split. The baseline requests 50% validation-cohort approval. Ten percent more approvals relative requests 55%; ten percentage points more requests 60%. Actual approval rates can differ due to threshold ties and cohort differences.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

# The executor supplies these paths. Interactive runs default to local exports.
ROOT = Path(os.environ.get('HOME_CREDIT_PROJECT_ROOT', str(Path.cwd()))).resolve()
if not (ROOT / 'src' / 'analysis.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import ensure_analysis, load_features, eda_tables

INPUT = Path(os.environ.get('HOME_CREDIT_FEATURES', str(ROOT / 'exports' / 'applicant_features.csv')))
OUTPUT = Path(os.environ.get('HOME_CREDIT_ANALYSIS', str(ROOT / 'outputs' / 'analysis')))
STATUS = os.environ.get('HOME_CREDIT_DATA_STATUS', 'real')
LGD = float(os.environ.get('HOME_CREDIT_LGD', '0.45'))
SEED = int(os.environ.get('HOME_CREDIT_SEED', '42'))
manifest = ensure_analysis(INPUT, OUTPUT, data_status=STATUS, lgd=LGD, seed=SEED)
get_ipython().run_line_magic('matplotlib', 'inline')
display(Markdown(f'**Data status: {manifest["data_status"].upper()}**. '
    + ('Synthetic pipeline validation; no Home Credit conclusions.' if STATUS != 'real'
       else 'Computed from the supplied accepted-cohort export.')))


In [ ]:
cutoffs = pd.read_csv(OUTPUT / 'cutoff_table.csv')
comparisons = pd.read_csv(OUTPUT / 'strategy_comparisons.csv')
display(cutoffs)
display(comparisons)
display(Image(filename=str(OUTPUT / 'figures' / 'strategy_tradeoff.png')))

## Answer: 10% more approvals

Compare the 50% and 55% requested policies with the fixed test thresholds. Approval/bad-rate intervals come from observed accepted-cohort outcomes; expected loss is a model-and-assumption scenario. No test-derived optimization or outcome-based acceptance decision is made.

In [ ]:
row = comparisons[(comparisons.scenario == '10_percent_more_approvals_relative') & (comparisons.split == 'test')].iloc[0]
display(Markdown(f'Fixed thresholds approve **{row.baseline_actual_approval_rate:.2%} → {row.scenario_actual_approval_rate:.2%}** on test. '
    f'Observed TARGET rate among approved applicants is **{row.baseline_bad_rate:.2%} → {row.scenario_bad_rate:.2%}** '
    f'(**{row.bad_rate_change_percentage_points:+.2f} percentage points**). '
    f'Scenario expected-loss change: **{row.expected_loss_change:,.2f} dataset currency units**.'))

## Reject inference and expected-loss assumptions

Only current accepted-applicant outcomes are observed. Previously rejected current applicants have no outcome, so these curves cannot say what happens when lending to them. Refusal history is a feature, not a substitute for missing rejected outcomes. Expected loss is PD × assumed LGD × EAD, using current credit as the exposure proxy. LGD is configurable (default 45%) and is not estimated from recoveries; execution preserves the completed analysis's LGD. TARGET records repayment difficulty, not confirmed loss. Known/missing EAD counts and EL coverage accompany each policy. Loss is a known-EAD subtotal when coverage is incomplete and is unavailable for a nonempty all-missing-EAD portfolio. The all-approved denominator and known-EAD-only denominator are labeled separately. Neither scenario EL nor its differences are accounting loss forecasts.

In [ ]:
display(Markdown((OUTPUT / 'findings.md').read_text()))